# KAN-REC — Análisis de cierre para la memoria

Este notebook produce las **métricas comprometidas en la propuesta de TFM**
que faltaban tras la comparativa principal. No forma parte del pipeline de
Fabric: opera sobre los checkpoints ya entrenados, así que no altera el
circuito productivo.

Cubre tres brechas frente a lo prometido:

| Prometido (propuesta) | Sección | Aquí |
|---|---|---|
| "mismo operador simbólico recuperado en ≥4 de 5 semillas" | 7.1 | Estabilidad entre 3 semillas |
| "Latencia (ms/batch) — media sobre 100 batches" | 7.2 | Latencia por encoder en GPU |
| "Violation rate monotonicity — coherencia de dominio" | 7.2 | Auditoría de monotonía |

Además reformula el objetivo de *gap < 0,01 RMSE*, que partía de una premisa
incorrecta (ver la celda final).

**Requisitos:** haber ejecutado el notebook de comparativa, de modo que
existan en `CKPT_DIR` los checkpoints `best_<encoder>_gs10_s<seed>.pt` de los
tres encoders y las tres semillas.

In [ ]:
get_ipython().system('pip install --quiet "git+https://github.com/bdm-lab-cap/kanrec.git@main"')
get_ipython().system('pip install --quiet scikit-learn')

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os

TSV_PATH    = "/content/drive/MyDrive/kanrec/criteo_10m.tsv"
CKPT_DIR    = "/content/drive/MyDrive/kanrec_checkpoints"
RESULTS_DIR = "/content/drive/MyDrive/kanrec_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

import glob
ckpts = sorted(glob.glob(f"{CKPT_DIR}/best_*_gs*_s*.pt"))
print(f"{len(ckpts)} checkpoints encontrados:")
for c in ckpts:
    print("  ", os.path.basename(c))
if not ckpts:
    raise FileNotFoundError(
        f"No hay checkpoints en {CKPT_DIR}. Ejecuta antes el notebook de comparativa."
    )

## 1. Datos y utilidades

Se reutiliza la misma ingesta que la comparativa (réplica verificada del
pipeline de Fabric) para que los análisis operen sobre exactamente la misma
distribución con la que se entrenaron los modelos.

In [ ]:
import glob, json, os, re, time
from collections import Counter

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

from kanrec.model import KANRecModel
from kanrec.baselines import build_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cpu":
    print("AVISO: sin GPU. La latencia medida NO sera comparable con la que "
          "promete la propuesta (T4). Activa GPU antes de continuar.")

NUMERICAL_COLS   = [f"I{i}" for i in range(1, 14)]
CATEGORICAL_COLS = [f"C{i}" for i in range(1, 27)]
LOG_COLS = [f"I{i}" for i in range(1, 6)]
STD_COLS = [f"I{i}" for i in range(6, 14)]
idx_cols = [f"{c}_idx" for c in CATEGORICAL_COLS]

N_SAMPLE = 300_000   # suficiente para latencia y ablacion; no se reentrena nada


def replicate_fabric_01(tsv_path, seed=42, n_sample=None):
    """Replica de fabric/01 (verificada celda a celda contra Spark)."""
    cols = ["label"] + NUMERICAL_COLS + CATEGORICAL_COLS
    df = pd.read_csv(tsv_path, sep="\t", header=None, names=cols,
                     na_values=[""], keep_default_na=True)
    if n_sample:
        df = df.sample(n=min(n_sample, len(df)), random_state=seed).reset_index(drop=True)
    for c in NUMERICAL_COLS:
        df[c] = df[c].fillna(0.0).abs()

    n = len(df)
    rng = np.random.RandomState(seed)
    perm = rng.permutation(n)
    n_tr, n_va = int(n * 0.8), int(n * 0.1)
    train_df = df.iloc[perm[:n_tr]].reset_index(drop=True)
    val_df   = df.iloc[perm[n_tr:n_tr + n_va]].reset_index(drop=True)
    test_df  = df.iloc[perm[n_tr + n_va:]].reset_index(drop=True)
    del df

    for d in (train_df, val_df, test_df):
        for c in LOG_COLS:
            d[c] = np.log1p(np.maximum(d[c].values, 0.0))
    means, stds = train_df[STD_COLS].mean(), train_df[STD_COLS].std(ddof=1)
    for d in (train_df, val_df, test_df):
        for c in STD_COLS:
            d[c] = (d[c] - means[c]) / stds[c]

    cardinalities = []
    for c in CATEGORICAL_COLS:
        counts = train_df[c].value_counts()
        ordered = sorted(counts.index, key=lambda v: (-counts[v], v))
        code_map = {v: i for i, v in enumerate(ordered)}
        invalid = len(ordered)
        cardinalities.append(invalid + 1)
        for d in (train_df, val_df, test_df):
            d[f"{c}_idx"] = d[c].map(code_map).fillna(invalid).astype("int64")
            d.drop(columns=[c], inplace=True)
    return train_df, val_df, test_df, cardinalities


print("Ingiriendo datos...")
t0 = time.time()
train_df, val_df, test_df, cat_cardinalities = replicate_fabric_01(TSV_PATH, n_sample=N_SAMPLE)
print(f"  {len(train_df):,} / {len(val_df):,} / {len(test_df):,} filas en {time.time()-t0:.0f}s")


def to_tensors(df):
    x_num = torch.tensor(df[NUMERICAL_COLS].fillna(0).values.astype("float32"))
    present = [c for c in idx_cols if c in df.columns]
    x_cat = torch.tensor(df[present].fillna(0).values.astype("int64"))
    y = torch.tensor(df["label"].values.astype("float32"))
    return x_num, x_cat, y

x_num_te, x_cat_te, y_te = to_tensors(test_df)
test_loader = DataLoader(TensorDataset(x_num_te, x_cat_te, y_te), batch_size=2048)


def load_checkpoint(path, encoder="kan-bspline", grid_size=10):
    """Reconstruye el modelo con la arquitectura correcta y carga los pesos."""
    sd = torch.load(path, map_location="cpu")
    cards = [sd[f"cat_embeddings.{i}.weight"].shape[0] - 1
             for i in range(len(CATEGORICAL_COLS))]
    m = build_model(encoder, num_numerical=len(NUMERICAL_COLS),
                    cat_cardinalities=cards, embedding_dim=16,
                    kan_grid_size=grid_size)
    m.load_state_dict(sd)
    m.eval()
    return m

## 2. Estabilidad de la extracción simbólica entre semillas

**Prometido en 7.1:** *"mismo operador simbólico recuperado en ≥4 de 5 semillas"*.

Se ejecuta la extracción sobre los tres checkpoints de KAN-REC (semillas 42,
123, 256) y se mide en cuántas de ellas cada campo converge al mismo operador.
Si tres modelos entrenados de forma independiente recuperan la misma forma
funcional, la extracción es reproducible y no un artefacto de una corrida.

El ajuste se hace sobre las 16 dimensiones del embedding, no solo la 0: una
sola proyección no basta para afirmar la forma de φⱼ.

In [ ]:
from scipy.optimize import curve_fit
from kanrec.symbolic import OPERATOR_LIBRARY, FORMULA_TEMPLATES

def fit_field_all_dims(model, field_idx, r2_threshold=0.90):
    """Ajusta la libreria de operadores a las 16 dimensiones de phi_j."""
    x_grid, curves = model.numerical_encoder.get_spline_curves(field_idx)
    x_np = x_grid.cpu().numpy()
    per_dim = []
    for d in range(curves.shape[1]):
        y_np = curves[:, d].cpu().numpy()
        best = {"operator": None, "r2": -1.0, "params": None, "formula": "?"}
        for name, fn in OPERATOR_LIBRARY.items():
            try:
                with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
                    params, _ = curve_fit(fn, x_np, y_np, maxfev=5000)
                    y_pred = fn(x_np, *params)
                if not np.all(np.isfinite(y_pred)) or not np.all(np.isfinite(params)):
                    continue
                ss_tot = np.sum((y_np - y_np.mean()) ** 2)
                if ss_tot < 1e-12:
                    continue
                r2 = float(1 - np.sum((y_np - y_pred) ** 2) / (ss_tot + 1e-10))
                if r2 > best["r2"]:
                    a, b = round(float(params[0]), 4), round(float(params[1]), 4)
                    best = {"operator": name, "r2": r2,
                            "params": [float(params[0]), float(params[1])],
                            "formula": FORMULA_TEMPLATES[name].format(a=a, b=b)}
            except Exception:
                continue
        if best["operator"]:
            per_dim.append(best)
    if not per_dim:
        return None
    ops = [f["operator"] for f in per_dim]
    dominant, n_dom = Counter(ops).most_common(1)[0]
    r2s = np.array([f["r2"] for f in per_dim])
    rep = max([f for f in per_dim if f["operator"] == dominant], key=lambda f: f["r2"])
    return {"operator": dominant, "params": rep["params"], "formula": rep["formula"],
            "accepted": bool(r2s.mean() >= r2_threshold),
            "r2_mean": float(r2s.mean()), "r2_std": float(r2s.std()),
            "r2_min": float(r2s.min()), "operator_agreement": n_dom / len(per_dim)}


kan_ckpts = sorted(glob.glob(f"{CKPT_DIR}/best_kan-bspline_gs10_s*.pt"))
print(f"Extrayendo de {len(kan_ckpts)} checkpoints de KAN-REC...\n")

results_by_seed = {}
for ck in kan_ckpts:
    seed = int(re.search(r"_s(\d+)\.pt$", ck).group(1))
    model = load_checkpoint(ck)
    norms = model.numerical_encoder.get_edge_norms()
    surviving = [j for j, n in enumerate(norms) if n >= np.percentile(norms, 20)]
    print(f"Seed {seed}: {len(surviving)}/{len(norms)} campos retenidos")
    seed_res = {}
    for j in surviving:
        r = fit_field_all_dims(model, j)
        if r:
            seed_res[NUMERICAL_COLS[j]] = r
    results_by_seed[seed] = seed_res
    print(f"  {sum(1 for r in seed_res.values() if r['accepted'])} formulas aceptadas")

In [ ]:
# Tabla de estabilidad
n_seeds = len(results_by_seed)
field_ops = {}
for seed, res in results_by_seed.items():
    for field, r in res.items():
        if r["accepted"]:
            field_ops.setdefault(field, []).append(r["operator"])

print(f"{'='*72}")
print(f"ESTABILIDAD DEL OPERADOR ENTRE {n_seeds} SEMILLAS")
print(f"{'='*72}")
print(f"{'campo':>6} {'operador':>10} {'semillas':>10} {'estabilidad':>12} {'R2 medio':>10}")
print("-" * 72)

stability_rows = []
for field in sorted(field_ops, key=lambda f: NUMERICAL_COLS.index(f)):
    ops = field_ops[field]
    dom, cnt = Counter(ops).most_common(1)[0]
    r2s = [results_by_seed[s][field]["r2_mean"] for s in results_by_seed
           if field in results_by_seed[s]]
    stability_rows.append({"field": field, "operator": dom, "seeds_agreeing": cnt,
                           "n_seeds": n_seeds, "stability": cnt / n_seeds,
                           "r2_mean": float(np.mean(r2s))})
    print(f"{field:>6} {dom:>10} {cnt:>7}/{n_seeds} {cnt/n_seeds:>11.0%} {np.mean(r2s):>10.4f}")

stability_df = pd.DataFrame(stability_rows)
if stability_df.empty:
    raise RuntimeError(
        "Ninguna formula supero el umbral de R2 en ninguna semilla. Revisa que "
        "los checkpoints correspondan a modelos ENTRENADOS y que el grid_size "
        "usado aqui (10) coincida con el del entrenamiento."
    )
perfect = (stability_df.stability == 1.0).sum()
print(f"\n{perfect}/{len(stability_df)} campos con operador identico en las {n_seeds} semillas")
print(f"Estabilidad media: {stability_df.stability.mean():.1%}")

stability_df.to_csv(f"{RESULTS_DIR}/estabilidad_semillas.csv", index=False)
print(f"\nGuardado en {RESULTS_DIR}/estabilidad_semillas.csv")

## 3. Latencia de inferencia

**Prometido en 7.2:** *"Latencia (ms/batch) — Eficiencia del encoder — Batch
size 4096 en GPU T4; media sobre 100 batches"*.

La propuesta afirmaba además que EfficientKAN sería *"igual o inferior"* en
latencia a AutoDis, al no requerir la suma ponderada de meta-embeddings. Esa
afirmación es contrastable y aquí se contrasta.

Se separa el tiempo del **encoder numérico** del tiempo del modelo completo,
porque es el encoder lo que se compara. Se descartan iteraciones de
calentamiento y se sincroniza CUDA antes de cada medición.

In [ ]:
from kanrec.latency import compare_latency

BATCH = 4096   # el tamano que fija la propuesta
x_num_b = x_num_te[:BATCH]
x_cat_b = x_cat_te[:BATCH]

models = {}
for enc in ["raw", "autodis", "kan-bspline"]:
    ck = f"{CKPT_DIR}/best_{enc}_gs10_s42.pt"
    if os.path.exists(ck):
        models[enc] = load_checkpoint(ck, encoder=enc).to(device)
    else:
        print(f"  (falta {os.path.basename(ck)}, se omite)")

latency = compare_latency(models, x_num_b, x_cat_b, n_warmup=10, n_runs=100)

lat_df = pd.DataFrame([
    {"encoder": k, "modelo_ms": v["full_ms_mean"], "modelo_ms_std": v["full_ms_std"],
     "encoder_ms": v["encoder_ms_mean"], "encoder_pct": v["encoder_share"],
     "filas_por_s": v["throughput_rows_per_s"], "batch": v["batch_size"],
     "device": v["device"]}
    for k, v in latency.items()
])
lat_df.to_csv(f"{RESULTS_DIR}/latencia.csv", index=False)
print(f"\nGuardado en {RESULTS_DIR}/latencia.csv")

if "kan-bspline" in latency and "autodis" in latency:
    ratio = latency["kan-bspline"]["full_ms_mean"] / latency["autodis"]["full_ms_mean"]
    print(f"\nContraste con la propuesta ('EfficientKAN igual o mas rapido que AutoDis'):")
    print(f"  KAN-REC / AutoDis = {ratio:.2f}x  ->  "
          f"{'CONFIRMADO' if ratio <= 1.05 else 'REFUTADO: KAN-REC es mas lento'}")

## 4. Auditoría de monotonía

**Prometido en 7.2:** *"Violation rate monotonicity — Coherencia de dominio —
% puntos donde la curva viola la monotonicidad esperada"*.

Un matiz necesario: en Criteo las variables I1–I13 son **anónimas**, así que no
existe una dirección "esperada" que pueda afirmarse sin inventarla. Se reporta
por tanto la monotonía **observada**: qué fracción de los tramos de cada curva
va en la dirección minoritaria. Cerca de 0 significa curva monótona; cerca de
0,5, sin dirección dominante.

Esto es auditable igualmente y es lo que un regulador querría comprobar: si el
modelo es monótono en una variable, esa propiedad se puede verificar y
declarar, aunque la variable esté anonimizada.

In [ ]:
def monotonicity_audit(model, field_indices=None, tol=1e-4):
    """Monotonia observada por campo, promediada sobre las 16 dimensiones."""
    if field_indices is None:
        field_indices = list(range(model.numerical_encoder.num_fields))
    rows = []
    for j in field_indices:
        _, curves = model.numerical_encoder.get_spline_curves(j)
        rates, dirs = [], []
        for d in range(curves.shape[1]):
            diffs = np.diff(curves[:, d].cpu().numpy())
            n_up, n_down = int((diffs > tol).sum()), int((diffs < -tol).sum())
            if n_up + n_down == 0:
                rates.append(0.0); dirs.append("flat"); continue
            rates.append(min(n_up, n_down) / len(diffs))
            dirs.append("creciente" if n_up >= n_down else "decreciente")
        rows.append({"field": NUMERICAL_COLS[j],
                     "violation_rate": float(np.mean(rates)),
                     "violation_std": float(np.std(rates)),
                     "direction": Counter(dirs).most_common(1)[0][0]})
    return pd.DataFrame(rows)


model42 = load_checkpoint(f"{CKPT_DIR}/best_kan-bspline_gs10_s42.pt")
mono_df = monotonicity_audit(model42)

print(f"{'campo':>6} {'direccion':>12} {'violacion':>12} {'veredicto':>16}")
print("-" * 52)
for _, r in mono_df.iterrows():
    v = ("monotona" if r.violation_rate < 0.05
         else "casi monotona" if r.violation_rate < 0.15 else "NO monotona")
    print(f"{r.field:>6} {r.direction:>12} {r.violation_rate:>10.2%}"
          f"±{r.violation_std:<5.2%} {v:>14}")

n_mono = (mono_df.violation_rate < 0.05).sum()
print(f"\n{n_mono}/{len(mono_df)} campos monotonos (violacion < 5%)")
print(f"Violacion media: {mono_df.violation_rate.mean():.2%}")

mono_df.to_csv(f"{RESULTS_DIR}/monotonia.csv", index=False)
print(f"\nGuardado en {RESULTS_DIR}/monotonia.csv")

## 5. Fidelidad simbólica y reformulación del objetivo de *gap*

**Prometido en 7.1:** *"gap de aproximación ‖ŷ_KAN − ŷ_sym‖ < 0,01 RMSE"*.

Ese objetivo partía de una **premisa incorrecta**, y conviene decirlo con
claridad en la memoria: asumía que la fórmula simbólica podía sustituir al
scoring completo. Pero la extracción opera sobre φⱼ, la función de
codificación *por campo*, mientras que el scoring pasa además por 26
embeddings categóricas, la capa de interacción y la cabeza. Un RMSE de 0,01
entre la fórmula y la predicción final no era alcanzable con esa arquitectura,
ni lo sería con ninguna: son objetos distintos.

La métrica correcta es la **ablación por sustitución**: reemplazar φⱼ por su
fórmula *dentro* del modelo, dejando el resto intacto, y medir (a) cuánto se
desvía la curva sustituida de la original y (b) cuánto se degrada el AUC.

In [ ]:
from kanrec.ablation import substitution_ablation

ablation_by_seed = {}
for seed, res in results_by_seed.items():
    ck = f"{CKPT_DIR}/best_kan-bspline_gs10_s{seed}.pt"
    model = load_checkpoint(ck).to(device)
    print(f"\n{'='*60}\nSeed {seed}\n{'='*60}")
    ablation_by_seed[seed] = substitution_ablation(
        model, test_loader, res, numerical_cols=NUMERICAL_COLS, device=device
    )

abl_df = pd.DataFrame([
    {"seed": s, "auc_original": a["auc_original"], "auc_sustituido": a["auc_substituted"],
     "delta_auc": a["delta_auc"], "error_curva_medio": a["curve_rel_error_mean"],
     "error_curva_max": a["curve_rel_error_max"], "rmse_predicciones": a["prediction_rmse"]}
    for s, a in ablation_by_seed.items()
])
print(f"\n{'='*72}\nFIDELIDAD SOBRE {len(abl_df)} SEMILLAS\n{'='*72}")
print(abl_df.to_string(index=False))
print(f"\nError de curva: {abl_df.error_curva_medio.mean():.3f} "
      f"± {abl_df.error_curva_medio.std():.3f}")
print(f"Delta AUC:      {abl_df.delta_auc.mean():+.4f} ± {abl_df.delta_auc.std():.4f}")

abl_df.to_csv(f"{RESULTS_DIR}/fidelidad.csv", index=False)
print(f"\nGuardado en {RESULTS_DIR}/fidelidad.csv")

## 6. Resumen para la memoria

In [ ]:
resumen = {
    "estabilidad": {
        "n_semillas": int(n_seeds),
        "campos_analizados": int(len(stability_df)),
        "campos_operador_identico": int((stability_df.stability == 1.0).sum()),
        "estabilidad_media": float(stability_df.stability.mean()),
    },
    "latencia": {k: {"modelo_ms": v["full_ms_mean"], "encoder_ms": v["encoder_ms_mean"],
                     "encoder_pct": v["encoder_share"],
                     "filas_por_s": v["throughput_rows_per_s"]}
                 for k, v in latency.items()},
    "monotonia": {
        "campos_monotonos": int((mono_df.violation_rate < 0.05).sum()),
        "campos_totales": int(len(mono_df)),
        "violacion_media": float(mono_df.violation_rate.mean()),
    },
    "fidelidad": {
        "error_curva_medio": float(abl_df.error_curva_medio.mean()),
        "error_curva_std": float(abl_df.error_curva_medio.std()),
        "delta_auc_medio": float(abl_df.delta_auc.mean()),
        "delta_auc_std": float(abl_df.delta_auc.std()),
    },
}

with open(f"{RESULTS_DIR}/resumen_cierre.json", "w") as f:
    json.dump(resumen, f, indent=2)

print("=" * 72)
print("RESUMEN PARA LA MEMORIA")
print("=" * 72)
print(json.dumps(resumen, indent=2, ensure_ascii=False))
print(f"\nGuardado en {RESULTS_DIR}/resumen_cierre.json")
print("\nFicheros generados en RESULTS_DIR:")
for f in ["estabilidad_semillas.csv", "latencia.csv", "monotonia.csv",
          "fidelidad.csv", "resumen_cierre.json"]:
    print(f"  - {f}")

## 7. Optimización: vectorización del encoder

El perfilado de la sección 3 identificó el cuello de botella: el encoder KAN
consume el **81,7%** del tiempo de inferencia, y el modelo es 4,3× más lento
que la normalización directa.

La causa no es que evaluar B-splines sea caro, sino que el `forward` recorre
los 13 campos en un **bucle de Python**: 13 lanzamientos de kernel CUDA
secuenciales por batch, cada uno con trabajo diminuto (una columna).

`VectorizedKANEncoder` los evalúa en un solo kernel usando `torch.bmm`,
manteniendo los campos independientes (no sirve un `KANLinear` de 13
entradas: su `F.linear` los mezclaría). La salida es **numéricamente
idéntica** —diferencia máxima medida 2,4e-7, precisión de float32—, así que
todos los resultados anteriores siguen aplicando.

**Importante:** en CPU esta versión es ~1,3× más *lenta* (sin coste de
lanzamiento que amortizar, el bucle gana). La hipótesis es que en GPU ocurre
lo contrario. Esta celda lo mide en lugar de suponerlo: si no gana, se
reporta como intento fallido.

In [ ]:
from kanrec.vectorized import vectorize_model

model_orig = load_checkpoint(f"{CKPT_DIR}/best_kan-bspline_gs10_s42.pt").to(device)
model_fast = vectorize_model(model_orig).to(device)

# Los tensores del batch se definieron en CPU en la seccion 3;
# compare_latency los movia internamente, pero aqui se usan directamente.
x_num_b = x_num_b.to(device)
x_cat_b = x_cat_b.to(device)

# 1. VERIFICAR EQUIVALENCIA antes de medir nada: si la salida cambia, la
#    "optimizacion" seria en realidad otro modelo y no serviria.
with torch.no_grad():
    a = model_orig(x_num_b, x_cat_b)
    b = model_fast(x_num_b, x_cat_b)
max_diff = (a - b).abs().max().item()
print(f"Diferencia maxima entre original y vectorizado: {max_diff:.2e}")
equivalent = torch.allclose(a, b, atol=1e-4)
print(f"Equivalencia numerica: {'OK' if equivalent else 'FALLA'}")
if not equivalent:
    raise RuntimeError("El encoder vectorizado NO es equivalente; no usar.")

# 2. Medir
vec_latency = compare_latency(
    {"kan-original": model_orig, "kan-vectorizado": model_fast},
    x_num_b, x_cat_b, n_warmup=10, n_runs=100,
)

speedup = (vec_latency["kan-original"]["full_ms_mean"]
           / vec_latency["kan-vectorizado"]["full_ms_mean"])
enc_speedup = (vec_latency["kan-original"]["encoder_ms_mean"]
               / vec_latency["kan-vectorizado"]["encoder_ms_mean"])

print(f"\nSpeedup modelo completo: {speedup:.2f}x")
print(f"Speedup solo encoder:    {enc_speedup:.2f}x")

if "raw" in latency:
    antes = latency["kan-bspline"]["full_ms_mean"] / latency["raw"]["full_ms_mean"]
    despues = vec_latency["kan-vectorizado"]["full_ms_mean"] / latency["raw"]["full_ms_mean"]
    print(f"\nCoste de la interpretabilidad frente a raw:")
    print(f"  antes de vectorizar:  {antes:.2f}x")
    print(f"  despues:              {despues:.2f}x")

if speedup > 1.05:
    print("\n-> La vectorizacion MEJORA en esta GPU. Reportar en 7.1 y 7.5.")
elif speedup > 0.95:
    print("\n-> Sin diferencia apreciable. Reportar como intento neutro en 7.5.")
else:
    print("\n-> La vectorizacion EMPEORA en esta GPU. Reportar como intento "
          "fallido en 7.5: el coste de los tensores mayores supera al ahorro "
          "de lanzamientos de kernel.")

resumen["vectorizacion"] = {
    "equivalencia_max_diff": float(max_diff),
    "speedup_modelo": float(speedup),
    "speedup_encoder": float(enc_speedup),
    "ms_original": vec_latency["kan-original"]["full_ms_mean"],
    "ms_vectorizado": vec_latency["kan-vectorizado"]["full_ms_mean"],
}
with open(f"{RESULTS_DIR}/resumen_cierre.json", "w") as f:
    json.dump(resumen, f, indent=2)
print(f"\nresumen_cierre.json actualizado")